# Task 4: Developing a Flask API for Deep Learning Models

## Objective
To expose trained deep learning models as REST APIs using the Flask framework, implementing robust JSON request-response handling, input schema validation, HTTP error handling, and comprehensive verification.

## Key Features & Deliverables
- **PyTorch Deep Learning Model**: Multi-layer Deep Neural Network (`DeepHealthRiskNet`) trained for clinical risk classification.
- **Flask REST API**: Production-grade Flask service exposing endpoints `/`, `/health`, `/predict`, and `/docs`.
- **JSON Request-Response Handling**: Single-instance, batch, and key-value dictionary feature payload support.
- **Exception & Error Handling**: Custom HTTP error responses for `400 Bad Request`, `404 Not Found`, `405 Method Not Allowed`, `415 Unsupported Media Type`, `422 Unprocessable Entity`, and `500 Internal Server Error`.
- **Automated Verification**: Interactive HTTP test client verifying 100% of test cases and visual probability plotting.

### Step 1: Environment Setup & Library Imports

In [ ]:
import os
import sys
import time
import json
import threading
from datetime import datetime, timezone
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from flask import Flask, request, jsonify, render_template_string
import requests
import matplotlib.pyplot as plt

print(f"PyTorch Version: {torch.__version__}")
print(f"NumPy Version: {np.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

### Step 2: PyTorch Deep Learning Model Architecture & Training
We define a 3-layer Neural Network with Batch Normalization, ReLU activations, and Dropout for clinical health risk classification (4 Classes: Low Risk, Moderate Risk, High Risk, Critical Risk).

In [ ]:
# 1. Define Model Architecture
class DeepHealthRiskNet(nn.Module):
    def __init__(self, input_dim=8, num_classes=4):
        super(DeepHealthRiskNet, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.network(x)

# 2. Dataset Generator
def generate_dataset(num_samples=2500):
    np.random.seed(42)
    age = np.random.uniform(18, 85, num_samples)
    bmi = np.random.uniform(16.0, 42.0, num_samples)
    bp = np.random.uniform(90, 180, num_samples)
    glucose = np.random.uniform(70, 240, num_samples)
    chol = np.random.uniform(130, 320, num_samples)
    hr = np.random.uniform(55, 110, num_samples)
    activity = np.random.uniform(0, 15, num_samples)
    sleep = np.random.uniform(4, 10, num_samples)
    X = np.column_stack([age, bmi, bp, glucose, chol, hr, activity, sleep])
    
    risk_score = (
        0.02 * age + 0.05 * (bmi - 22) + 0.03 * (bp - 120) + 
        0.04 * (glucose - 100) + 0.02 * (chol - 200) + 0.01 * (hr - 70) - 
        0.08 * activity - 0.05 * sleep + np.random.normal(0, 0.5, num_samples)
    )
    y = np.zeros(num_samples, dtype=np.int64)
    q33, q66, q90 = np.percentile(risk_score, [33, 66, 90])
    y[risk_score >= q33] = 1
    y[risk_score >= q66] = 2
    y[risk_score >= q90] = 3
    return X, y

# 3. Model Training & Export
X_raw, y_raw = generate_dataset(3000)
mean, std = np.mean(X_raw, axis=0), np.std(X_raw, axis=0) + 1e-8
X_scaled = (X_raw - mean) / std

split = int(0.8 * len(X_scaled))
train_x, val_x = torch.tensor(X_scaled[:split], dtype=torch.float32), torch.tensor(X_scaled[split:], dtype=torch.float32)
train_y, val_y = torch.tensor(y_raw[:split], dtype=torch.long), torch.tensor(y_raw[split:], dtype=torch.long)

model = DeepHealthRiskNet(8, 4)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

loader = DataLoader(TensorDataset(train_x, train_y), batch_size=64, shuffle=True)
for epoch in range(1, 31):
    model.train()
    for bx, by in loader:
        optimizer.zero_grad()
        loss = criterion(model(bx), by)
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    preds = model(val_x)
    acc = (preds.argmax(dim=1) == val_y).float().mean().item() * 100
print(f"PyTorch Model Training Complete! Validation Accuracy: {acc:.2f}%")

# Save Weights & Config
os.makedirs("saved_models", exist_ok=True)
torch.save(model.state_dict(), "saved_models/dl_model.pt")
with open("saved_models/config.json", "w") as f:
    json.dump({
        "model_name": "DeepHealthRiskNet",
        "input_dim": 8,
        "num_classes": 4,
        "feature_names": ["age", "bmi", "blood_pressure", "glucose_level", "cholesterol", "heart_rate", "physical_activity_hours", "sleep_hours"],
        "class_labels": {"0": "Low Risk", "1": "Moderate Risk", "2": "High Risk", "3": "Critical Risk"},
        "mean": mean.tolist(),
        "std": std.tolist()
    }, f, indent=4)
print("Model weights & configuration metadata saved successfully.")

### Step 3: Flask REST API Definition & Exception Handlers
We import `app.py` and run tests against the endpoints.

In [ ]:
from app import app
client = app.test_client()

# Test /health
r_health = client.get("/health")
print("Health Endpoint Response:")
print(json.dumps(r_health.get_json(), indent=2))

# Test /predict (Valid Single Sample)
sample_payload = {"features": [52.0, 28.4, 138.0, 145.0, 235.0, 80.0, 3.0, 6.5]}
r_pred = client.post("/predict", json=sample_payload)
print("\nPrediction Endpoint Response:")
print(json.dumps(r_pred.get_json(), indent=2))

### Step 4: Verification of HTTP Error Handling Scenarios

In [ ]:
print("--- Testing Error Scenarios ---")

# 1. HTTP 415 Unsupported Media Type
r_415 = client.post("/predict", headers={"Content-Type": "text/plain"}, data="raw_data")
print(f"1. Unsupported Media Type -> Status {r_415.status_code}: {r_415.get_json()['error_code']}")

# 2. HTTP 400 Bad Request (Missing Key)
r_400 = client.post("/predict", json={"bad_key": [1, 2, 3]})
print(f"2. Missing Required Key -> Status {r_400.status_code}: {r_400.get_json()['error_code']}")

# 3. HTTP 422 Unprocessable Entity (Wrong Dimension)
r_422 = client.post("/predict", json={"features": [52.0, 28.4, 138.0]})
print(f"3. Wrong Feature Dimension -> Status {r_422.status_code}: {r_422.get_json()['error_code']}")

# 4. HTTP 405 Method Not Allowed
r_405 = client.get("/predict")
print(f"4. Method Not Allowed -> Status {r_405.status_code}: {r_405.get_json()['error_code']}")

# 5. HTTP 404 Not Found
r_404 = client.get("/non_existent_path")
print(f"5. Not Found -> Status {r_404.status_code}: {r_404.get_json()['error_code']}")

### Step 5: Visualizing Deep Learning Prediction Probabilities

In [ ]:
res = r_pred.get_json()["predictions"][0]
class_probs = res["class_probabilities"]

plt.figure(figsize=(9, 5))
bars = plt.bar(class_probs.keys(), class_probs.values(), color=['#22c55e', '#eab308', '#f97316', '#ef4444'])
plt.title(f"Deep Learning Health Risk Probabilities\nPredicted Label: {res['predicted_label']} (Confidence: {res['confidence_score']*100:.1f}%)", fontsize=12, fontweight='bold')
plt.xlabel("Risk Category")
plt.ylabel("Softmax Probability")
plt.ylim(0, 1.0)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.02, f"{yval*100:.1f}%", ha='center', va='bottom', fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()